# 4.6.- Taller práctico de Redes Neuronales Artificiales (ANN)

Este notebook está diseñado como una **clase práctica progresiva** para consolidar los conceptos del Tema 4 de ANN y enlazar con los notebooks 4.1–4.5.

### Objetivos de la sesión
- Entender qué puede y qué no puede resolver un **perceptrón**.
- Visualizar y comparar **sigmoide, tanh, ReLU y Leaky ReLU**.
- Construir una **MLP con Keras** para clasificación no lineal.
- Interpretar **loss, accuracy, épocas, batch size y learning rate**.
- Detectar **underfitting y overfitting**.
- Aplicar **EarlyStopping, Dropout y BatchNormalization**.
- Resolver un problema **multiclase con Softmax**.
- Comparar configuraciones de **hiperparámetros** y guardar/cargar el mejor modelo.

**Duración orientativa:** 90–120 minutos.  
**Nivel:** guiado al principio y cada vez más autónomo.


## 0. Preparación

Ejecuta esta celda antes de comenzar. Se fijan semillas para que los resultados sean reproducibles.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

from sklearn.datasets import make_moons, make_blobs
from sklearn.linear_model import Perceptron
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

np.random.seed(42)
tf.random.set_seed(42)

print('TensorFlow:', tf.__version__)


---
## Ejercicio 1 — De TLU a perceptrón: puertas lógicas

Una TLU calcula una suma ponderada y aplica un **umbral**. En clase se ha visto que puede representar funciones lógicas linealmente separables como AND, OR y NAND.

### 1A. Completa una TLU genérica
Implementa una función `TLU(x, w, theta)` que devuelva 1 cuando $w^T x \geq 	heta$ y 0 en caso contrario.


In [ ]:
def TLU(x, w, theta=0):
    # TODO: calcula la suma ponderada y aplica el umbral
    pass

entradas = [(0,0), (0,1), (1,0), (1,1)]

# TODO: elige pesos y umbral para AND
w_and = [...] 
theta_and = ...
print('AND:', [TLU(x, w_and, theta_and) for x in entradas])

# TODO: repite para OR y NAND


### 1B. Pregunta de razonamiento
¿Puedes encontrar **una única recta** que separe correctamente las salidas 0 y 1 de XOR? Justifica tu respuesta antes de pasar al siguiente ejercicio.


---
## Ejercicio 2 — Perceptrón de Rosenblatt y la limitación de XOR

Entrena un `Perceptron` de scikit-learn para AND, OR y XOR. Compara las predicciones.


In [ ]:
X_logic = np.array([[0,0],[0,1],[1,0],[1,1]])
y_and = np.array([0,0,0,1])
y_or  = np.array([0,1,1,1])
y_xor = np.array([0,1,1,0])

# TODO: entrena un Perceptron para cada puerta lógica y muestra predicciones + accuracy


**Interpreta:** si XOR no llega al 100%, no es un fallo de entrenamiento: el problema no es linealmente separable. Esta limitación motiva el uso de **capas ocultas + activaciones no lineales**.


---
## Ejercicio 3 — Funciones de activación y sus derivadas

Representa en el mismo rango las funciones:
- Sigmoide
- tanh
- ReLU
- Leaky ReLU ($lpha=0.1$)

Después representa también sus derivadas aproximadas o analíticas. El objetivo es relacionar la forma de la derivada con el **vanishing gradient**.


In [ ]:
x = np.linspace(-6, 6, 500)

# TODO: calcula las cuatro funciones
sigmoid = ...
tanh = ...
relu = ...
leaky_relu = ...

# TODO: dibuja las cuatro curvas

# TODO: calcula/dibuja sus derivadas


### Preguntas rápidas
1. ¿En qué zonas la derivada de la sigmoide es casi 0?  
2. ¿Por qué eso puede ralentizar el aprendizaje en capas iniciales?  
3. ¿Qué ventaja aporta Leaky ReLU frente a ReLU cuando $x<0$?


---
## Ejercicio 4 — De frontera lineal a MLP: resolver `make_moons`

Crearemos un dataset no lineal. Primero probaremos un modelo sin capa oculta y después una MLP.


In [ ]:
X, y = make_moons(n_samples=1200, noise=0.22, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

plt.figure(figsize=(6,5))
plt.scatter(X_train[:,0], X_train[:,1], c=y_train, s=12, cmap='coolwarm')
plt.title('Dataset make_moons')
plt.xlabel('x1'); plt.ylabel('x2'); plt.show()


### 4A. Modelo lineal
Crea una red con **una sola neurona sigmoide** de salida. Es equivalente a una frontera lineal.


In [ ]:
# TODO: modelo con una única Dense(1, activation='sigmoid')
linear_model = ...

# TODO: compile con binary_crossentropy y accuracy
# TODO: fit durante 40 épocas
# TODO: evalúa en test


### 4B. MLP
Añade al menos **dos capas ocultas con ReLU**. Mantén una única salida sigmoide.


In [ ]:
# TODO: construye una MLP 2 -> 16 -> 16 -> 1
mlp = ...

# TODO: compila y entrena con validation_split=0.2
history = ...

# TODO: evalúa en test


### 4C. Curvas de aprendizaje
Representa `loss` y `val_loss`. Identifica si el modelo todavía está aprendiendo, se ha estabilizado o empieza a sobreajustar.


In [ ]:
# TODO: dibuja history.history['loss'] y history.history['val_loss']


---
## Ejercicio 5 — Learning rate: demasiado pequeño, adecuado o demasiado grande

Entrena la **misma arquitectura** con tres tasas de aprendizaje:
- `0.0001`
- `0.001`
- `0.1`

Compara la pérdida de validación. Mantén todo lo demás constante.


In [ ]:
def train_with_lr(lr):
    # TODO: crea exactamente la misma red del ejercicio anterior
    # TODO: usa Adam(learning_rate=lr)
    # TODO: devuelve el objeto history
    pass

# TODO: entrena con 1e-4, 1e-3 y 1e-1 y compara val_loss en un gráfico


**Discusión:** explica qué síntomas observas cuando el `learning_rate` es demasiado pequeño o demasiado grande. Relaciónalo con la idea de convergencia/inestabilidad vista en clase.


---
## Ejercicio 6 — Overfitting y regularización

Vamos a reducir el conjunto de entrenamiento para hacer más fácil que una red grande memorice los datos. Compararemos:
1. Red grande sin regularización.
2. Red con `Dropout`, `BatchNormalization` y `EarlyStopping`.


In [ ]:
# Subconjunto pequeño para provocar sobreajuste con más facilidad
X_small = X_train[:180]
y_small = y_train[:180]


In [ ]:
# TODO 1: crea una red grande (por ejemplo 3 capas de 128 neuronas) y entrénala 250 épocas
# TODO 2: crea otra con BatchNormalization + Dropout(0.35)
# TODO 3: usa EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)
# TODO 4: compara loss/val_loss de ambos modelos
# TODO 5: compara accuracy en test


### Preguntas de interpretación
- ¿En qué momento aparece una separación clara entre `loss` y `val_loss`?
- ¿EarlyStopping se activa antes de 250 épocas?
- ¿Dropout hace que la pérdida de entrenamiento sea necesariamente menor? ¿Por qué no?


---
## Ejercicio 7 — Clasificación multiclase con Softmax

Construye un clasificador para **3 clases**. La capa de salida debe tener 3 neuronas con `softmax`.


In [ ]:
X3, y3 = make_blobs(
    n_samples=1200, centers=3, n_features=2,
    cluster_std=2.0, random_state=7
)
X3_train, X3_test, y3_train, y3_test = train_test_split(
    X3, y3, test_size=0.25, stratify=y3, random_state=42
)
scaler3 = StandardScaler()
X3_train = scaler3.fit_transform(X3_train)
X3_test = scaler3.transform(X3_test)


In [ ]:
# TODO: modelo 2 -> 16 -> 3 con softmax
# Usa sparse_categorical_crossentropy porque y3 contiene etiquetas 0,1,2 (no one-hot)

softmax_model = ...

# TODO: entrena y calcula probabilidades para las primeras 5 muestras
# Comprueba que cada fila suma aproximadamente 1
# TODO: crea la matriz de confusión


---
## Ejercicio 8 — Mini búsqueda de hiperparámetros

Sin usar todavía Keras Tuner, prueba varias configuraciones y quédate con la que obtenga mejor `val_accuracy`.

Compara:
- neuronas: `8`, `16`, `32`
- activación: `'relu'`, `'tanh'`
- batch size: `16`, `64`

Son **12 combinaciones**. Usa pocas épocas + EarlyStopping para que sea rápido.


In [ ]:
results = []

# TODO: recorre units, activation y batch_size
# Para cada combinación:
#   1) crea la red
#   2) entrena con EarlyStopping
#   3) guarda el máximo val_accuracy y los hiperparámetros
# Finalmente ordena de mejor a peor


### Pregunta
¿Por qué una búsqueda de hiperparámetros debe comparar modelos usando **validación** y no usando directamente el conjunto de test?


---
## Ejercicio 9 — Guardar y cargar un modelo

Guarda el modelo multiclase del ejercicio 7 en formato `.keras`, vuelve a cargarlo y comprueba que produce las mismas probabilidades para una muestra.


In [ ]:
# TODO: guarda softmax_model en 'ann_softmax_ejercicio.keras'
# TODO: cárgalo con keras.models.load_model
# TODO: compara la predicción original y la del modelo cargado


---
# Reto final — Diseña tu propia red

Sin copiar una arquitectura anterior, intenta mejorar el resultado sobre `make_moons`.

Puedes modificar:
- número de capas ocultas;
- neuronas por capa;
- activación;
- `learning_rate`;
- `batch_size`;
- `Dropout`;
- `BatchNormalization`;
- `EarlyStopping`.

### Condición
No uses el conjunto de test para tomar decisiones durante el diseño. Úsalo **solo al final**.

Anota:
1. arquitectura final;
2. hiperparámetros elegidos;
3. mejor `val_accuracy`;
4. `test_accuracy` final;
5. una explicación de 4–5 líneas justificando tus decisiones.


In [ ]:
# TU PROPUESTA
final_model = ...


## Checklist final de aprendizaje

Al terminar deberías poder explicar con tus propias palabras:
- por qué XOR no se resuelve con un único perceptrón;
- para qué sirve una función de activación no lineal;
- diferencia entre sigmoide y softmax;
- qué hacen `loss`, optimizador y backpropagation durante el entrenamiento;
- qué significan época y batch size;
- cómo reconocer overfitting en las curvas;
- para qué sirven EarlyStopping, Dropout y BatchNormalization;
- qué hiperparámetros se pueden ajustar y por qué el test no debe usarse para elegirlos.
